# Cataract Detection - ResNet50 Training

This notebook trains a ResNet50 image classifier for cataract detection using a pre-split, preprocessed image dataset. It saves the trained model and label map so the chatbot backend can classify user-uploaded eye images.

## Expected dataset layout

Set `DATASET_DIR` to your dataset root. The notebook expects your 70/15/15 split to already exist on disk:

```text
cataract_dataset/
  train/
    cataract/
    normal/
  validation/    # val/ also works
    cataract/
    normal/
  test/
    cataract/
    normal/
```

The notebook reads your prepared train, validation, and test folders directly.

In [ ]:
# Run this once if TensorFlow is not installed in your notebook environment.
# %pip install tensorflow matplotlib scikit-learn

In [ ]:
from pathlib import Path
import json
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow:', tf.__version__)

In [ ]:
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

# Change this path if your pre-split dataset lives somewhere else.
DATASET_DIR = Path(os.getenv('DATASET_DIR', 'cataract_dataset'))
TRAIN_DIR = DATASET_DIR / 'train'
VAL_DIR = DATASET_DIR / 'validation'
if not VAL_DIR.exists():
    VAL_DIR = DATASET_DIR / 'val'
TEST_DIR = DATASET_DIR / 'test'

MODEL_DIR = Path('models')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'cataract_resnet50.keras'
LABELS_PATH = MODEL_DIR / 'cataract_labels.json'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

if not DATASET_DIR.exists():
    raise FileNotFoundError(
        f'Dataset directory not found: {DATASET_DIR.resolve()}\n'
        'Update DATASET_DIR to your cataract_dataset folder.'
    )

for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if not split_dir.exists():
        raise FileNotFoundError(f'Missing split folder: {split_dir.resolve()}')

class_names = sorted([p.name for p in TRAIN_DIR.iterdir() if p.is_dir()])
if len(class_names) < 2:
    raise ValueError('Expected at least 2 class folders inside the train split.')

print('Dataset:', DATASET_DIR.resolve())
print('Train split:', TRAIN_DIR.resolve())
print('Validation split:', VAL_DIR.resolve())
print('Test split:', TEST_DIR.resolve())
print('Classes:', class_names)

In [ ]:
train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

val_ds = keras.utils.image_dataset_from_directory(
    VAL_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
)

# Keep the class order that Keras used and save it for chatbot inference.
class_names = train_ds.class_names
num_classes = len(class_names)
LABELS_PATH.write_text(json.dumps({'class_names': class_names}, indent=2))
print('Class index map:', {i: name for i, name in enumerate(class_names)})

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(10, 6))
for images, labels in train_ds.take(1):
    for i in range(min(8, images.shape[0])):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[int(labels[i])])
        plt.axis('off')
plt.tight_layout()

In [ ]:
def compute_class_weights(dataset, class_count):
    counts = np.zeros(class_count, dtype=np.int64)
    for _, labels in dataset.unbatch():
        counts[int(labels.numpy())] += 1
    total = counts.sum()
    weights = {i: total / (class_count * count) for i, count in enumerate(counts) if count > 0}
    return counts, weights

class_counts, class_weight = compute_class_weights(train_ds, num_classes)
print('Training image counts:', dict(zip(class_names, class_counts.tolist())))
print('Class weights:', class_weight)

In [ ]:
base_model = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SIZE + (3,))
x = preprocess_input(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)

outputs = layers.Dense(num_classes, activation='softmax', name='class_probabilities')(x)
loss = 'sparse_categorical_crossentropy'
metrics = ['accuracy']

model = keras.Model(inputs, outputs, name='cataract_resnet50')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss,
    metrics=metrics,
)
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)

model.save(MODEL_PATH)
print('Saved model to:', MODEL_PATH.resolve())
print('Saved labels to:', LABELS_PATH.resolve())

In [ ]:
history_df = {
    key: [float(v) for v in values]
    for key, values in history.history.items()
}
(MODEL_DIR / 'training_history.json').write_text(json.dumps(history_df, indent=2))

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(np.argmax(preds, axis=1).tolist())

print(classification_report(y_true, y_pred, target_names=class_names))
print('Confusion matrix:')
print(confusion_matrix(y_true, y_pred))

## Optional fine-tuning

If the validation metrics look stable, unfreeze the last ResNet blocks and train for a few more epochs with a very small learning rate. For a strict 10-epoch run, skip this section.

In [ ]:
# base_model.trainable = True
# for layer in base_model.layers[:-30]:
#     layer.trainable = False
#
# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-5),
#     loss=loss,
#     metrics=metrics,
# )
# fine_tune_history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=3,
#     class_weight=class_weight,
#     callbacks=callbacks,
# )
# model.save(MODEL_PATH)

## Chatbot upload inference helper

Use the same preprocessing for uploaded images in the chatbot backend. The function below returns the predicted label and confidence.

In [ ]:
def predict_uploaded_image(image_path, model_path=MODEL_PATH, labels_path=LABELS_PATH):
    loaded_model = keras.models.load_model(model_path)
    labels = json.loads(Path(labels_path).read_text())['class_names']

    img = keras.utils.load_img(image_path, target_size=IMG_SIZE)
    arr = keras.utils.img_to_array(img)
    batch = np.expand_dims(arr, axis=0)
    preds = loaded_model.predict(batch, verbose=0)

    class_idx = int(np.argmax(preds[0]))
    confidence = float(preds[0][class_idx])

    return {
        'label': labels[class_idx],
        'confidence': round(confidence, 4),
        'class_index': class_idx,
    }

# Example after training:
# predict_uploaded_image('path/to/uploaded_eye_image.jpg')